# POLITE — Observatory Control

Interactive night operation: **PlaneWave mount** (PWI4) + **QHY268M** / **ZWO EFW**
(Alpaca) + **Pyxis HWP**. Phased like the scripted night sessions — startup,
calibration, imaging, shutdown — but live, one cell at a time.

**Kernel:** `POLITE` conda environment.

**Before you start**
- **PWI4** is running and reachable (`localhost:8220`).
- The Alpaca server (INDIGO agent / ASCOM Remote) exposes camera, wheel, and — if
  `ALPACA_CONFIG.rotator_index` is set — the Pyxis HWP as an Alpaca rotator.
- **One kernel owns the hardware.** Restart the kernel to fully reset.

WARNING: Cells in the Startup and Imaging phases command **physical motion** (homing,
slewing, HWP rotation). Read each before running.

In [ ]:
# --- POLITE path bootstrap ---
# Run from the repo root so `from obs_utils import ...` resolves. Jupyter starts
# the kernel in this notebook's own directory, so hop up one level when needed.
import os, sys
from pathlib import Path
_cwd = Path.cwd()
if _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
_root = str(Path.cwd())
if _root not in sys.path:
    sys.path.insert(0, _root)
print("POLITE root:", _root)

## Phase 1 · Startup

`startup_observatory` connects + enables + **homes** the mount (physical motion),
loads the pointing model, and connects the instrument. `adopt_startup` hands those
live connections to the interactive helper so the `s.filter / s.hwp / s.expose`
methods work here too.

In [ ]:
from obs_utils.startup import StartupConfig, startup_observatory
from obs_utils import user_config as uc
from obs_utils import interactive as obs

cfg = StartupConfig(
    pwi4=uc.PWI4_CONFIG,
    alpaca=uc.ALPACA_CONFIG,
    pointing_model_filename="DefaultModel.pxp",
)
state = startup_observatory(cfg)   # WARNING: homes the mount
s = obs.adopt_startup(state)
s.status()

## Phase 1b · Per-component connect  *(fault-isolated fallback)*

`startup_observatory` above is all-or-nothing: mount + pointing model + instrument
in one call, so a single offline device blocks the rest. When that happens, bring
each component up on its own with the cells below — each attaches to the same
shared session `s`, and a failure in one cell does not stop the others. Run only
the cells you need; the `s.filter / s.hwp / s.expose / s.status` helpers work on
whatever connected.

Note: `connect_mount()` only creates the PWI4 client — it does **not** enable,
home, slew, or load a pointing model. Do the mount startup deliberately (Phase 1
or the explicit PWI4 calls) once the client is up.

In [ ]:
s = obs.connect_camera()          # QHY268M only

In [ ]:
s = obs.connect_filter_wheel()    # ZWO EFW only

In [ ]:
s = obs.connect_rotator()         # Alpaca HWP rotator only (needs rotator_index set)

In [ ]:
s = obs.connect_mount()           # PWI4 client only — no motion, no pointing model
s.status()

## Phase 2 · Calibration

Take calibration frames at the current pointing. Example: a short bias ramp on the
Dark slot.

In [ ]:
from pathlib import Path
outdir = Path("data/calib"); outdir.mkdir(parents=True, exist_ok=True)

s.filter("Dark")
for i in range(5):
    s.expose(0.0, out_path=outdir / f"bias_{i:03d}.fits", dark=True)

## Phase 3 · Imaging

Slew, choose a filter, expose. Fill in your target coordinates.

In [ ]:
from obs_utils.mount import slew_radec_j2000, wait_for_slew

RA_HOURS = 5.9195   # <-- target RA  [hours]
DEC_DEG  = 7.407    # <-- target Dec [deg]

slew_radec_j2000(s.pwi4, RA_HOURS, DEC_DEG)   # WARNING: slews the mount
wait_for_slew(s.pwi4)
s.status()

In [ ]:
from pathlib import Path
outdir = Path("data/lights"); outdir.mkdir(parents=True, exist_ok=True)

s.filter("Photometric V")
s.expose(30.0, out_path=outdir / "target_V_030s.fits")

### Phase 3b · Polarimetry sequence *(optional)*

Step the HWP through modulation angles, one frame each. Uses the Alpaca rotator
when `rotator_index` is set.

In [ ]:
from pathlib import Path
outdir = Path("data/pol"); outdir.mkdir(parents=True, exist_ok=True)

for ang in (0.0, 22.5, 45.0, 67.5):
    s.hwp(ang)
    s.expose(30.0, out_path=outdir / f"pol_hwp{ang:04.1f}.fits")

## Phase 4 · Shutdown

`obs.shutdown()` releases the **instrument only** — the mount is intentionally left
untouched. Park/stop the mount deliberately when the night is done.

In [ ]:
# Mount teardown is deliberate (not a side effect of clearing the session):
s.pwi4.mount_tracking_off()
s.pwi4.mount_park()
# s.pwi4.mount_disconnect()   # uncomment to fully disconnect the mount

obs.shutdown()   # releases camera / wheel / rotator